# 1.3 · Tokens, contexto e imágenes

**Objetivo:** gestionar la ventana de contexto con criterio y ver que la multimodalidad es solo otro tipo de bloque (con su coste).

**Al terminar tendrás:** las funciones `tokens()`, `reset()` e `imagen()`, y una celda final que convierte el chat en un CLI interactivo.

## Preparación

Partimos de lo que construimos en el 1.2: `SYSTEM`, `messages` y `chat()` con streaming.

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

import anthropic
client = anthropic.Anthropic()   # lee ANTHROPIC_API_KEY del entorno

In [ ]:
SYSTEM = (
    "Eres un agente de soporte de nivel 1 de una empresa de software. "
    "Respondes en español, en tono cercano y como máximo en dos frases."
)

In [ ]:
messages = []

def chat(pregunta):
    messages.append({"role": "user", "content": pregunta})
    with client.messages.stream(
        model=MODEL, max_tokens=4096, system=SYSTEM, messages=messages,
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
        final = stream.get_final_message()
    messages.append({"role": "assistant", "content": final.content})
    print(f"\n[input_tokens={final.usage.input_tokens}]")

Un par de turnos para tener historial:

In [ ]:
chat("Hola, soy Ana. No puedo descargar la factura 1234 en PDF")

In [ ]:
chat("Me sale un error 500 cada vez que lo intento")

## Contar antes de enviar

`count_tokens` dice cuántos tokens de entrada ocuparía una petición **sin enviarla** (y sin generar respuesta).

| | Cuándo | Para qué |
|---|---|---|
| `count_tokens` | **antes** de enviar | decidir si recortar, avisar, elegir estrategia |
| `usage` | **después** de enviar | saber lo que realmente se ha consumido |

In [ ]:
def tokens():
    count = client.messages.count_tokens(
        model=MODEL, system=SYSTEM, messages=messages
    )
    return count.input_tokens

tokens()

## `max_tokens` y `stop_reason`

`max_tokens` es un **tope duro** de salida. Si el modelo lo alcanza, la respuesta queda **cortada**, no resumida.

In [ ]:
# ❌ FALLO PROVOCADO
r = client.messages.create(
    model=MODEL,
    max_tokens=40,
    system=SYSTEM,
    messages=[{
        "role": "user",
        "content": "Explícame paso a paso cómo resetear una contraseña",
    }],
)
r.stop_reason, r.content

`stop_reason == "max_tokens"`: hay que comprobarlo siempre.

Fíjate en `content`: puede que solo haya un bloque `thinking`, o un texto a medias. El razonamiento también cuenta como salida, así que con un tope tan bajo el modelo puede gastarlo entero pensando y no llegar a escribir nada.

Añadimos el aviso a `chat()`:

In [ ]:
def chat(pregunta):
    messages.append({"role": "user", "content": pregunta})
    with client.messages.stream(
        model=MODEL, max_tokens=4096, system=SYSTEM, messages=messages,
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
        final = stream.get_final_message()
    messages.append({"role": "assistant", "content": final.content})
    if final.stop_reason == "max_tokens":
        print("\n⚠️  Respuesta truncada por max_tokens")
    print(f"\n[input_tokens={final.usage.input_tokens}]")

## Estrategia de recorte: ventana deslizante

Nos quedamos con los últimos `MAX_TURNOS` turnos (cada turno = un mensaje `user` y uno `assistant`).

Regla de la API: **el historial debe empezar por un mensaje `user`**. Si el corte cae en un `assistant`, lo descartamos.

In [ ]:
MAX_TURNOS = 10

def recortar():
    global messages
    recorte = messages[-MAX_TURNOS * 2:]
    # el historial debe empezar por un user
    while recorte and recorte[0]["role"] != "user":
        recorte = recorte[1:]
    messages = recorte

def reset():
    messages.clear()

Para que se note el recorte, bajamos `MAX_TURNOS` a 1 (solo el último turno):

In [ ]:
MAX_TURNOS = 1
print("antes:", len(messages), "mensajes,", tokens(), "tokens")
recortar()
print("después:", len(messages), "mensajes,", tokens(), "tokens")
MAX_TURNOS = 10

### Ventana deslizante frente a resumir

| | Ventana deslizante | Resumir lo antiguo |
|---|---|---|
| Cómo | se descartan los turnos más viejos | un modelo resume los turnos viejos en un mensaje |
| Gana | simple, gratis, predecible | conserva lo importante del principio |
| Pierde | olvida el contexto inicial del caso (¡el nombre de Ana!) | cuesta una llamada extra y el resumen puede perder detalles |

En el bloque 3 veremos que el Agent SDK **compacta automáticamente**. Ahora ya sabéis qué problema resuelve.

## Multimodalidad

Una imagen es **otro tipo de bloque** dentro de `content`: en vez de un string, el mensaje lleva una lista con un bloque `image` (en base64) y un bloque `text`.

La función `imagen()` muestra la captura en el notebook y añade el mensaje al historial (todavía sin enviarlo).

In [ ]:
import base64, mimetypes
from IPython.display import Image, display

def imagen(ruta, texto):
    display(Image(filename=ruta, width=400))         # la vemos en el notebook
    media_type = mimetypes.guess_type(ruta)[0]
    with open(ruta, "rb") as f:
        data = base64.standard_b64encode(f.read()).decode("utf-8")
    messages.append({"role": "user", "content": [
        {"type": "image", "source": {
            "type": "base64", "media_type": media_type, "data": data,
        }},
        {"type": "text", "text": texto},
    ]})

¿Cuánto cuesta una imagen? Contamos antes y después de añadirla:

In [ ]:
tokens()

In [ ]:
imagen(
    "../../datos/captura_error.png",
    "El cliente adjunta esta captura. ¿Qué error ves y qué le dirías?",
)

In [ ]:
tokens()

**La imagen cuesta tokens** y se queda en el historial: se paga **en cada turno siguiente**.

Para enviar este turno partimos `chat()` en dos: `enviar()` manda lo que haya en `messages` y guarda la respuesta; `chat()` solo añade el texto y llama a `enviar()`.

In [ ]:
def enviar():
    with client.messages.stream(
        model=MODEL, max_tokens=4096, system=SYSTEM, messages=messages,
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
        final = stream.get_final_message()
    messages.append({"role": "assistant", "content": final.content})
    if final.stop_reason == "max_tokens":
        print("\n⚠️  Respuesta truncada por max_tokens")
    print(f"\n[input_tokens={final.usage.input_tokens}]")

def chat(pregunta):
    messages.append({"role": "user", "content": pregunta})
    enviar()

In [ ]:
enviar()

`content` puede ser un string o una **lista de bloques**: texto, imagen, documento… Es la misma estructura que vimos en la respuesta del 1.1.

In [ ]:
[b["type"] for b in messages[-2]["content"]]

In [ ]:
chat("¿Y qué significa ese código de error?")

## Del notebook al CLI

Todo lo que hemos construido celda a celda cabe en un bucle. Esto es un CLI conversacional: si lo pasas a un `.py`, lo tienes en terminal.

Comandos: `/tokens`, `/reset` y `salir` para terminar.

In [ ]:
# escribe "salir" para terminar
while (pregunta := input("tú> ")) != "salir":
    if pregunta == "/tokens": print(tokens()); continue
    if pregunta == "/reset":  reset(); continue
    chat(pregunta)
    print()

## Resumen del bloque

1. La API es *stateless*: el historial (`messages`) es tu responsabilidad.
2. Cada turno reenvía y paga todo lo anterior.
3. Todo son bloques de contenido: texto, imagen y, en el próximo bloque, **herramientas**.

## 🔸 Reto

Haz que `reset()` conserve el primer mensaje del usuario (el contexto del caso) y borre el resto.